In [ ]:
import os, random, csv
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, cohen_kappa_score
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

# -----------------------
# Configartion
# -----------------------
IMG_DIR = "images"
ANN_DIR = "annotations"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4
VAL_SPLIT = 0.2
LOSS_WEIGHTS = {"cls": 2.0, "val": 0.5, "aro": 0.5}
LOG_FILE = "training_log.csv"

# -----------------------
# Seed
# -----------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# -----------------------
# Dataset
# -----------------------
class EmotionDataset(Dataset):
    def __init__(self, img_dir=IMG_DIR, ann_dir=ANN_DIR, transform=None):
        self.img_dir = Path(img_dir)
        self.ann_dir = Path(ann_dir)
        self.ids = sorted([p.stem for p in self.img_dir.glob("*") if p.suffix.lower() in (".jpg", ".png")])
        self.transform = transform

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        _id = self.ids[idx]
        img_path = self.img_dir / f"{_id}.jpg"
        if not img_path.exists():
            img_path = self.img_dir / f"{_id}.png"
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)

        exp = int(np.load(self.ann_dir / f"{_id}_exp.npy"))
        val = float(np.load(self.ann_dir / f"{_id}_val.npy"))
        aro = float(np.load(self.ann_dir / f"{_id}_aro.npy"))

        return {
            "id": _id,
            "image": img,
            "expression": torch.tensor(exp, dtype=torch.long),
            "valence": torch.tensor(val, dtype=torch.float32),
            "arousal": torch.tensor(aro, dtype=torch.float32)
        }

# -----------------------
# Model
# -----------------------
class ResNetMultiHead(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        base = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        feat_dim = 512
        self.expr_head = nn.Linear(feat_dim, num_classes)
        self.val_head = nn.Linear(feat_dim, 1)
        self.aro_head = nn.Linear(feat_dim, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        return {
            "logits": self.expr_head(feat),
            "valence": self.val_head(feat).squeeze(1),
            "arousal": self.aro_head(feat).squeeze(1)
        }

# -----------------------
# Metrics
# -----------------------
def rmse(pred, target):
    return np.sqrt(np.mean((pred - target) ** 2))

def corr(pred, target):
    return pearsonr(pred, target)[0] if len(set(target)) > 1 else 0.0

def sagr(pred, target):
    return np.mean(np.sign(pred) == np.sign(target))

def ccc(pred, target):
    mean_pred, mean_true = np.mean(pred), np.mean(target)
    var_pred, var_true = np.var(pred), np.var(target)
    cov = np.mean((pred-mean_pred)*(target-mean_true))
    return (2*cov) / (var_pred + var_true + (mean_pred-mean_true)**2 + 1e-8)

def accuracy(logits, targets):
    return (torch.argmax(logits, dim=1) == targets).float().mean().item()

# -----------------------
# Evaluate
# -----------------------
def evaluate(model, loader, device, cls_loss_fn, reg_loss_fn):
    model.eval()
    total_loss, total_acc, n = 0, 0, 0
    all_preds, all_true = [], []
    all_val_pred, all_val_true = [], []
    all_aro_pred, all_aro_true = [], []

    with torch.no_grad():
        for batch in loader:
            imgs = batch["image"].to(device)
            expr = batch["expression"].to(device)
            vals = batch["valence"].to(device)
            aros = batch["arousal"].to(device)

            out = model(imgs)
            loss_cls = cls_loss_fn(out["logits"], expr)
            loss_val = reg_loss_fn(out["valence"], vals)
            loss_aro = reg_loss_fn(out["arousal"], aros)
            loss = LOSS_WEIGHTS["cls"]*loss_cls + LOSS_WEIGHTS["val"]*loss_val + LOSS_WEIGHTS["aro"]*loss_aro

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_acc += accuracy(out["logits"], expr) * bs
            n += bs

            all_preds.extend(torch.argmax(out["logits"],1).cpu().numpy())
            all_true.extend(expr.cpu().numpy())
            all_val_pred.extend(out["valence"].cpu().numpy())
            all_val_true.extend(vals.cpu().numpy())
            all_aro_pred.extend(out["arousal"].cpu().numpy())
            all_aro_true.extend(aros.cpu().numpy())

    all_preds, all_true = np.array(all_preds), np.array(all_true)
    all_val_pred, all_val_true = np.array(all_val_pred), np.array(all_val_true)
    all_aro_pred, all_aro_true = np.array(all_aro_pred), np.array(all_aro_true)

    f1 = f1_score(all_true, all_preds, average="weighted")
    kappa = cohen_kappa_score(all_true, all_preds)

    metrics = {
        "loss": total_loss/n,
        "acc": total_acc/n,
        "f1": f1,
        "kappa": kappa,
        "val_rmse": rmse(all_val_pred, all_val_true),
        "val_corr": corr(all_val_pred, all_val_true),
        "val_sagr": sagr(all_val_pred, all_val_true),
        "val_ccc": ccc(all_val_pred, all_val_true),
        "aro_rmse": rmse(all_aro_pred, all_aro_true),
        "aro_corr": corr(all_aro_pred, all_aro_true),
        "aro_sagr": sagr(all_aro_pred, all_aro_true),
        "aro_ccc": ccc(all_aro_pred, all_aro_true)
    }
    return metrics

# -----------------------
# Training Loop with Logging
# -----------------------
def train():
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    full_ds = EmotionDataset(transform=None)
    n = len(full_ds)
    val_size = int(VAL_SPLIT * n)
    train_size = n - val_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])

    train_ds = [(dict(item, image=train_tf(item["image"]))) for item in train_ds]
    val_ds   = [(dict(item, image=val_tf(item["image"]))) for item in val_ds]

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False, pin_memory=True)

    model = ResNetMultiHead().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    cls_loss_fn = nn.CrossEntropyLoss()
    reg_loss_fn = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_val_acc = -1
    logs = []

    # write CSV header
    with open(LOG_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch","train_loss","train_acc","val_loss","val_acc","f1","kappa",
            "val_rmse","val_corr","val_sagr","val_ccc",
            "aro_rmse","aro_corr","aro_sagr","aro_ccc"
        ])

    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss, total_acc, n = 0, 0, 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            imgs = batch["image"].to(DEVICE)
            expr = batch["expression"].to(DEVICE)
            vals = batch["valence"].to(DEVICE)
            aros = batch["arousal"].to(DEVICE)

            out = model(imgs)
            loss_cls = cls_loss_fn(out["logits"], expr)
            loss_val = reg_loss_fn(out["valence"], vals)
            loss_aro = reg_loss_fn(out["arousal"], aros)
            loss = LOSS_WEIGHTS["cls"]*loss_cls + LOSS_WEIGHTS["val"]*loss_val + LOSS_WEIGHTS["aro"]*loss_aro

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_acc += accuracy(out["logits"], expr) * bs
            n += bs

        train_loss = total_loss/n
        train_acc = total_acc/n
        val_metrics = evaluate(model, val_loader, DEVICE, cls_loss_fn, reg_loss_fn)
        scheduler.step(val_metrics["loss"])

        # save log row
        with open(LOG_FILE, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch, train_loss, train_acc,
                val_metrics["loss"], val_metrics["acc"], val_metrics["f1"], val_metrics["kappa"],
                val_metrics["val_rmse"], val_metrics["val_corr"], val_metrics["val_sagr"], val_metrics["val_ccc"],
                val_metrics["aro_rmse"], val_metrics["aro_corr"], val_metrics["aro_sagr"], val_metrics["aro_ccc"]
            ])

        print(f"Epoch {epoch}/{EPOCHS} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} "
              f"| Val Acc {val_metrics['acc']:.4f} F1 {val_metrics['f1']:.3f} Kappa {val_metrics['kappa']:.3f}")

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            torch.save(model.state_dict(), f"best_resnet18.pt")
            print("✅ Saved best model")

    print(f"Training complete. Best Val Acc: {best_val_acc:.4f}")

if __name__ == "__main__":
    print("Device:", DEVICE)
    train()


Device: cuda


/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/30: 100%|██████████| 100/100 [00:08<00:00, 11.22it/s]


Epoch 1/30 | Train Loss 3.8519 Acc 0.3056 | Val Acc 0.3617 F1 0.354 Kappa 0.273
✅ Saved best model


Epoch 2/30: 100%|██████████| 100/100 [00:08<00:00, 11.17it/s]


Epoch 2/30 | Train Loss 2.1345 Acc 0.6984 | Val Acc 0.4105 F1 0.402 Kappa 0.327
✅ Saved best model


...

Epoch 28/30 | Train Loss 0.0174 Acc 1.0000 | Val Acc 0.4168 F1 0.413 Kappa 0.333


Epoch 29/30: 100%|██████████| 100/100 [00:10<00:00,  9.92it/s]


Epoch 29/30 | Train Loss 0.0148 Acc 1.0000 | Val Acc 0.4168 F1 0.413 Kappa 0.333


Epoch 30/30: 100%|██████████| 100/100 [00:10<00:00,  9.90it/s]


Epoch 30/30 | Train Loss 0.0158 Acc 1.0000 | Val Acc 0.4230 F1 0.422 Kappa 0.341
Training complete. Best Val Acc: 0.4305


In [5]:
# train_vgg_resnet.py
import os, random, csv
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, cohen_kappa_score
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

# -----------------------
# Config
# -----------------------
IMG_DIR = "images"
ANN_DIR = "annotations"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4
VAL_SPLIT = 0.2
LOSS_WEIGHTS = {"cls": 2.0, "val": 0.5, "aro": 0.5}

# -----------------------
# Seed
# -----------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

# -----------------------
# Dataset
# -----------------------
class EmotionDataset(Dataset):
    def __init__(self, img_dir=IMG_DIR, ann_dir=ANN_DIR, transform=None):
        self.img_dir = Path(img_dir)
        self.ann_dir = Path(ann_dir)
        self.ids = sorted([p.stem for p in self.img_dir.glob("*") if p.suffix.lower() in (".jpg", ".png")])
        self.transform = transform

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        _id = self.ids[idx]
        img_path = self.img_dir / f"{_id}.jpg"
        if not img_path.exists():
            img_path = self.img_dir / f"{_id}.png"
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)

        exp = int(np.load(self.ann_dir / f"{_id}_exp.npy"))
        val = float(np.load(self.ann_dir / f"{_id}_val.npy"))
        aro = float(np.load(self.ann_dir / f"{_id}_aro.npy"))

        return {
            "id": _id,
            "image": img,
            "expression": torch.tensor(exp, dtype=torch.long),
            "valence": torch.tensor(val, dtype=torch.float32),
            "arousal": torch.tensor(aro, dtype=torch.float32)
        }

# -----------------------
# Models
# -----------------------
class ResNetMultiHead(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        base = models.resnet18(pretrained=True)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        feat_dim = 512
        self.expr_head = nn.Linear(feat_dim, num_classes)
        self.val_head  = nn.Linear(feat_dim, 1)
        self.aro_head  = nn.Linear(feat_dim, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        return {
            "logits": self.expr_head(feat),
            "valence": self.val_head(feat).squeeze(1),
            "arousal": self.aro_head(feat).squeeze(1)
        }

class VGGMultiHead(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        base = models.vgg16(pretrained=True)
        self.backbone = base.features
        self.avgpool = base.avgpool
        self.fc = base.classifier[:-1]   # drop final 1000-class layer
        feat_dim = 4096
        self.expr_head = nn.Linear(feat_dim, num_classes)
        self.val_head  = nn.Linear(feat_dim, 1)
        self.aro_head  = nn.Linear(feat_dim, 1)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        feat = self.fc(feat)
        return {
            "logits": self.expr_head(feat),
            "valence": self.val_head(feat).squeeze(1),
            "arousal": self.aro_head(feat).squeeze(1)
        }

# -----------------------
# Metrics
# -----------------------
def rmse(pred, target):
    return np.sqrt(np.mean((pred - target) ** 2))

def corr(pred, target):
    return pearsonr(pred, target)[0] if len(set(target)) > 1 else 0.0

def sagr(pred, target):
    return np.mean(np.sign(pred) == np.sign(target))

def ccc(pred, target):
    mean_pred, mean_true = np.mean(pred), np.mean(target)
    var_pred, var_true = np.var(pred), np.var(target)
    cov = np.mean((pred-mean_pred)*(target-mean_true))
    return (2*cov) / (var_pred + var_true + (mean_pred-mean_true)**2 + 1e-8)

def accuracy(logits, targets):
    return (torch.argmax(logits, dim=1) == targets).float().mean().item()

# -----------------------
# Evaluate
# -----------------------
def evaluate(model, loader, device, cls_loss_fn, reg_loss_fn):
    model.eval()
    total_loss, total_acc, n = 0, 0, 0
    all_preds, all_true = [], []
    all_val_pred, all_val_true = [], []
    all_aro_pred, all_aro_true = [], []

    with torch.no_grad():
        for batch in loader:
            imgs = batch["image"].to(device)
            expr = batch["expression"].to(device)
            vals = batch["valence"].to(device)
            aros = batch["arousal"].to(device)

            out = model(imgs)
            loss_cls = cls_loss_fn(out["logits"], expr)
            loss_val = reg_loss_fn(out["valence"], vals)
            loss_aro = reg_loss_fn(out["arousal"], aros)
            loss = LOSS_WEIGHTS["cls"]*loss_cls + LOSS_WEIGHTS["val"]*loss_val + LOSS_WEIGHTS["aro"]*loss_aro

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_acc += accuracy(out["logits"], expr) * bs
            n += bs

            all_preds.extend(torch.argmax(out["logits"],1).cpu().numpy())
            all_true.extend(expr.cpu().numpy())
            all_val_pred.extend(out["valence"].cpu().numpy())
            all_val_true.extend(vals.cpu().numpy())
            all_aro_pred.extend(out["arousal"].cpu().numpy())
            all_aro_true.extend(aros.cpu().numpy())

    return {
        "loss": total_loss/n,
        "acc": total_acc/n,
        "f1": f1_score(all_true, all_preds, average="weighted"),
        "kappa": cohen_kappa_score(all_true, all_preds),
        "val_rmse": rmse(np.array(all_val_pred), np.array(all_val_true)),
        "val_corr": corr(np.array(all_val_pred), np.array(all_val_true)),
        "val_sagr": sagr(np.array(all_val_pred), np.array(all_val_true)),
        "val_ccc": ccc(np.array(all_val_pred), np.array(all_val_true)),
        "aro_rmse": rmse(np.array(all_aro_pred), np.array(all_aro_true)),
        "aro_corr": corr(np.array(all_aro_pred), np.array(all_aro_true)),
        "aro_sagr": sagr(np.array(all_aro_pred), np.array(all_aro_true)),
        "aro_ccc": ccc(np.array(all_aro_pred), np.array(all_aro_true)),
    }

# -----------------------
# Training Loop
# -----------------------
def run_model(model_name, model_class):
    print(f"\n🚀 Training {model_name.upper()}...\n")
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    full_ds = EmotionDataset(transform=None)
    n = len(full_ds)
    val_size = int(VAL_SPLIT * n)
    train_size = n - val_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])

    train_ds = [(dict(item, image=train_tf(item["image"]))) for item in train_ds]
    val_ds   = [(dict(item, image=val_tf(item["image"]))) for item in val_ds]

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False, pin_memory=True)

    model = model_class().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    cls_loss_fn = nn.CrossEntropyLoss()
    reg_loss_fn = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_val_acc = -1
    log_file = f"{model_name}_training_log.csv"

    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch","train_loss","train_acc","val_loss","val_acc","f1","kappa",
            "val_rmse","val_corr","val_sagr","val_ccc",
            "aro_rmse","aro_corr","aro_sagr","aro_ccc"
        ])

    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss, total_acc, n = 0, 0, 0
        for batch in tqdm(train_loader, desc=f"{model_name.upper()} Epoch {epoch}/{EPOCHS}"):
            imgs = batch["image"].to(DEVICE)
            expr = batch["expression"].to(DEVICE)
            vals = batch["valence"].to(DEVICE)
            aros = batch["arousal"].to(DEVICE)

            out = model(imgs)
            loss_cls = cls_loss_fn(out["logits"], expr)
            loss_val = reg_loss_fn(out["valence"], vals)
            loss_aro = reg_loss_fn(out["arousal"], aros)
            loss = LOSS_WEIGHTS["cls"]*loss_cls + LOSS_WEIGHTS["val"]*loss_val + LOSS_WEIGHTS["aro"]*loss_aro

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            bs = imgs.size(0)
            total_loss += loss.item() * bs
            total_acc += accuracy(out["logits"], expr) * bs
            n += bs

        train_loss = total_loss/n
        train_acc = total_acc/n
        val_metrics = evaluate(model, val_loader, DEVICE, cls_loss_fn, reg_loss_fn)
        scheduler.step(val_metrics["loss"])

        with open(log_file, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch, train_loss, train_acc,
                val_metrics["loss"], val_metrics["acc"], val_metrics["f1"], val_metrics["kappa"],
                val_metrics["val_rmse"], val_metrics["val_corr"], val_metrics["val_sagr"], val_metrics["val_ccc"],
                val_metrics["aro_rmse"], val_metrics["aro_corr"], val_metrics["aro_sagr"], val_metrics["aro_ccc"]
            ])

        print(f"{model_name.upper()} Epoch {epoch}/{EPOCHS} | "
              f"Train Acc {train_acc:.4f} | Val Acc {val_metrics['acc']:.4f} "
              f"F1 {val_metrics['f1']:.3f} Kappa {val_metrics['kappa']:.3f}")

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            torch.save(model.state_dict(), f"best_{model_name}.pt")
            print(f"✅ Saved best {model_name} model")

    print(f"✅ {model_name.upper()} training complete. Best Val Acc: {best_val_acc:.4f}")

# -----------------------
# Run both models
# -----------------------
if __name__ == "__main__":
    print("Device:", DEVICE)
    run_model("vgg", VGGMultiHead)
    run_model("resnet", ResNetMultiHead)

Device: cuda

🚀 Training VGG...



/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kintopi/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/kintopi/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [02:00<00:00, 4.60MB/s] 
VGG Epoch 1/30: 100%|██████████| 100/100 [00:45<00:00,  2.19it/s]


VGG Epoch 1/30 | Train Acc 0.1559 | Val Acc 0.2215 F1 0.131 Kappa 0.117
✅ Saved best vgg model


...

RESNET Epoch 28/30 | Train Acc 1.0000 | Val Acc 0.4230 F1 0.419 Kappa 0.339


RESNET Epoch 29/30: 100%|██████████| 100/100 [00:09<00:00, 10.28it/s]


RESNET Epoch 29/30 | Train Acc 1.0000 | Val Acc 0.4305 F1 0.429 Kappa 0.348


RESNET Epoch 30/30: 100%|██████████| 100/100 [00:09<00:00, 10.43it/s]


RESNET Epoch 30/30 | Train Acc 1.0000 | Val Acc 0.4293 F1 0.426 Kappa 0.346
✅ RESNET training complete. Best Val Acc: 0.4418
